# Configurable QLoRA Fine-Tuning

Both a one-call launcher and manual steps are available. Model IDs, paths, tokenizer options, and architecture-specific adapter settings live in ignored `config.local.json`, not in this notebook. Start from `config.example.json` when configuring a new experiment.

Select a compatible Python environment. Clear cell outputs before publication. Supported text-model architectures depend on the installed training libraries; changing configuration does not guarantee support for every architecture or fit on every GPU.

## 1. Configuration — manual path

The shortcut below can run without this cell. For manual operation, configure your local JSON file and enable LOAD_MODEL/RUN_TRAINING here. Do not run both entry paths for the same trial.

In [ ]:
from pathlib import Path
from project_config import load_config, runtime_paths, validate_training_config

PROJECT = Path.cwd()
CONFIG = load_config(PROJECT)
PATHS = runtime_paths(CONFIG, PROJECT)
MODEL_ID = CONFIG['model']['id']
MODEL_DIR = PATHS['model']
REVISION_FILE = PATHS['revision']
TRAIN_FILE = PATHS['train']
VALIDATION_FILE = PATHS['validation']
TEST_FILE = PATHS['test']
OUTPUT_DIR = PATHS['output']
TRAIN_CONFIG = CONFIG['training']
TEMPLATE_KWARGS = CONFIG['model']['chat_template_kwargs']
MAX_STEPS = TRAIN_CONFIG['max_steps']
MAX_SEQ_LENGTH = TRAIN_CONFIG['max_seq_length']
LORA_RANK = TRAIN_CONFIG['lora_rank']
SEED = TRAIN_CONFIG['seed']
DASHBOARD_URL = f"http://127.0.0.1:{CONFIG['dashboard_port']}"

DOWNLOAD_MODEL = False
LOAD_MODEL = False
RUN_TRAINING = False
if RUN_TRAINING:
    validate_training_config(CONFIG)
    if not LOAD_MODEL:
        raise ValueError('Enable LOAD_MODEL before manual training.')
print('Configuration ready.')


## Run all steps — optional shortcut

After Restart Kernel, run only the next Code cell. It validates the configured data, starts or reuses the dashboard, loads the local checkpoint, prepares the adapter, trains, validates, and saves automatically. The local configuration supplies all defaults. Open the printed dashboard URL during preparation.

Each invocation saves a separate run directory. A ten-step default is a smoke test, not ten epochs. Skip this shortcut if using the manual sections. Running the entire notebook executes this shortcut once and disables subsequent manual model/training flags.

In [ ]:
from pathlib import Path
from importlib import reload
import training_utils
reload(training_utils)

result = training_utils.run_finetuning(
    project=Path.cwd(),
    kernel_namespace=globals(),
)
print('Saved adapter:', result['adapter_path'])


## 2. Optional checkpoint download

Download only when needed. Pin the exact upstream revision. Model selection comes from local configuration; this cell embeds no model identifier or credentials.

In [ ]:
if DOWNLOAD_MODEL:
    from huggingface_hub import HfApi, snapshot_download
    if not MODEL_ID:
        raise ValueError('Set model.id in local configuration before downloading.')
    if REVISION_FILE.exists():
        provenance = dict(line.split('=', 1) for line in REVISION_FILE.read_text().splitlines() if '=' in line)
        if provenance.get('model_id') != MODEL_ID:
            raise ValueError('Checkpoint provenance and configured model ID differ.')
        revision = provenance['revision']
    else:
        revision = HfApi().model_info(MODEL_ID).sha
    snapshot_download(repo_id=MODEL_ID, revision=revision, local_dir=str(MODEL_DIR))
    REVISION_FILE.parent.mkdir(parents=True, exist_ok=True)
    REVISION_FILE.write_text(f'model_id={MODEL_ID}\nrevision={revision}\n', encoding='utf-8')
    print('Checkpoint download complete.')
else:
    print('Download skipped.')


## 3. Curated datasets

Each JSONL row contains conversational `messages`, ending in a verified assistant answer. Optional source metadata supports auditing. Keep train, validation, and final test distinct, including source groups and semantic near-duplicates. The basic helper checks exact duplicates only. Model-generated baseline responses are not verified training targets.

In [ ]:
from pathlib import Path
from project_config import load_config, runtime_paths
from training_utils import load_examples, check_split_overlap, format_examples

PROJECT = Path(globals().get('PROJECT', Path.cwd()))
CONFIG = globals().get('CONFIG', load_config(PROJECT))
PATHS = runtime_paths(CONFIG, PROJECT)
TRAIN_FILE, VALIDATION_FILE, TEST_FILE = PATHS['train'], PATHS['validation'], PATHS['test']
datasets_ready = False
train_examples = load_examples(TRAIN_FILE)
validation_examples = load_examples(VALIDATION_FILE)
test_examples = load_examples(TEST_FILE)
check_split_overlap(train_examples, validation_examples)
check_split_overlap(train_examples, test_examples)
check_split_overlap(validation_examples, test_examples)
datasets_ready = True
print(f'Training examples: {len(train_examples)}')
print(f'Validation examples: {len(validation_examples)}')
print(f'Held-out test examples: {len(test_examples)}')
print('Dataset checks passed. No training has started.')


## 4. Local model loading

Import Unsloth before training libraries. Restart the kernel instead of loading a second model into the same session. Free GPU memory and architecture support determine whether loading succeeds.

In [ ]:
if LOAD_MODEL:
    if globals().get('model') is not None:
        raise RuntimeError('A model is retained. Restart Kernel before loading again.')
    import unsloth
    import torch
    from unsloth import FastLanguageModel
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable.')
    if not list(MODEL_DIR.glob('*.safetensors')):
        raise FileNotFoundError('Download the configured checkpoint first.')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(MODEL_DIR), max_seq_length=MAX_SEQ_LENGTH, dtype=None,
        load_in_4bit=CONFIG['model']['load_in_4bit'], full_finetuning=False,
        local_files_only=True, device_map={'': torch.cuda.current_device()},
    )
    print('Local model loaded.')
else:
    print('GPU loading skipped.')


## 5. Freeze a baseline

Use `chat.py` for base-model conversations and `chat.py --adapter latest` for a saved adapter. Compare identical prompts and generation settings. Keep final-test judgments separate from training loss and token prediction accuracy. Final factual evaluation is not performed by this notebook.

## 6. Adapter preparation and training

For manual operation, start `.venv/bin/python monitoring.py` in a separate terminal and open its printed URL. Configure tokenizer role markers and supported LoRA target modules locally. Check loss masking before training. The dashboard must show this run's ready state.

In [ ]:
if RUN_TRAINING:
    validate_training_config(CONFIG)
    if not globals().get('datasets_ready'):
        raise RuntimeError('Validate datasets first.')
    if getattr(model, 'peft_config', None):
        raise RuntimeError('An adapter is already attached. Restart Kernel for a fresh run.')
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only
    from monitoring import create_training_callback
    train_texts = format_examples(train_examples, tokenizer, MAX_SEQ_LENGTH, TEMPLATE_KWARGS)
    validation_texts = format_examples(validation_examples, tokenizer, MAX_SEQ_LENGTH, TEMPLATE_KWARGS)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0,
        target_modules=TRAIN_CONFIG['lora_target_modules'], bias='none',
        use_gradient_checkpointing='unsloth', random_state=SEED,
    )
    model.print_trainable_parameters()
    trainer = SFTTrainer(
        model=model, processing_class=tokenizer,
        train_dataset=Dataset.from_dict({'text': train_texts}),
        eval_dataset=Dataset.from_dict({'text': validation_texts}),
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / 'checkpoints'), dataset_text_field='text',
            max_length=MAX_SEQ_LENGTH, packing=False, dataset_num_proc=1,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            gradient_accumulation_steps=TRAIN_CONFIG['gradient_accumulation_steps'], max_steps=MAX_STEPS,
            learning_rate=TRAIN_CONFIG['learning_rate'], warmup_steps=max(1, MAX_STEPS // 10), optim='adamw_8bit',
            bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=1, logging_nan_inf_filter=False, include_num_input_tokens_seen='non_padding',
            eval_strategy='steps', eval_steps=MAX_STEPS, save_strategy='no', report_to='none', seed=SEED,
        ),
    )
    trainer = train_on_responses_only(trainer, instruction_part=TRAIN_CONFIG['instruction_marker'], response_part=TRAIN_CONFIG['response_marker'])
    for split in (trainer.train_dataset, trainer.eval_dataset):
        if any(not any(label != -100 for label in row['labels']) for row in split):
            raise ValueError('An example has no assistant tokens contributing to loss.')
    monitor_callback = create_training_callback(OUTPUT_DIR / 'monitor', len(train_examples), model, max_steps=MAX_STEPS)
    trainer.add_callback(monitor_callback)
    print(f'Trainer prepared. Open {DASHBOARD_URL}.')
else:
    print('Trainer preparation skipped.')


In [ ]:
if RUN_TRAINING:
    if 'monitor_callback' not in globals() or monitor_callback not in trainer.callback_handler.callbacks:
        raise RuntimeError('Prepare the live monitor first.')
    if trainer.state.global_step > 0:
        raise RuntimeError('This Trainer already ran. Restart Kernel for a fresh run.')
    import json
    from urllib.request import urlopen
    with urlopen(DASHBOARD_URL + '/api/status', timeout=3) as response:
        dashboard_state = json.load(response)['state']
    if dashboard_state.get('run_id') != monitor_callback.log.state['run_id']:
        raise RuntimeError('The dashboard is following another run. Check its configured log directory.')
    torch.cuda.reset_peak_memory_stats()
    try:
        training_result = trainer.train()
        evaluation_result = next((row for row in reversed(trainer.state.log_history) if 'eval_loss' in row and row.get('step') == trainer.state.global_step), None)
        if evaluation_result is None:
            evaluation_result = trainer.evaluate()
        monitor_callback.set_status('evaluated')
    except BaseException as exc:
        monitor_callback.set_status('interrupted' if isinstance(exc, KeyboardInterrupt) else 'error', error=str(exc))
        raise
    print('Training metrics:', training_result.metrics)
    print('Validation metrics:', evaluation_result)
else:
    print('Training skipped.')


## 7. Save and test

The shortcut saves automatically in a unique run folder. This manual cell saves under the configured output directory. An adapter requires its original base checkpoint. Keep weights, local configuration, data, and logs out of Git; publish tested artifacts separately with their required model provenance.

In [ ]:
if RUN_TRAINING:
    import json
    import hashlib
    adapter_dir = OUTPUT_DIR / 'adapter'
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    trainer.save_metrics('train', training_result.metrics)
    trainer.save_metrics('eval', evaluation_result)
    metadata = {
        'model_id': MODEL_ID, 'config': CONFIG,
        'checkpoint_provenance': REVISION_FILE.read_text() if REVISION_FILE.exists() else 'not recorded',
        'max_steps': MAX_STEPS, 'max_seq_length': MAX_SEQ_LENGTH,
        'dataset_hashes': {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in (TRAIN_FILE, VALIDATION_FILE, TEST_FILE)},
        'final_test_evaluated': False,
    }
    (OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    monitor_callback.set_status('saved', adapter_path=str(adapter_dir.relative_to(PROJECT)))
    print('Adapter saved. Test with chat.py --adapter latest.')
else:
    print('No manual training artifacts to save.')
